In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Estimation of Obesity Levels Based On Eating Habits and Physical Condition

# Motivation

Obesity is a major public health issue with serious long-term consequences for cardiovascular health, metabolic disorders, and quality of life. Early identification of individuals at risk of obesity can help guide preventive interventions, lifestyle counseling, and resource allocation.

In this project, I use supervised machine learning to predict an individual’s obesity level using their demographic, behavioral, and lifestyle characteristics. The dataset includes 2,111 individuals from Mexico, Peru, and Colombia and contains information on age, height, weight, family history of overweight, eating habits, physical activity, technology use, and transportation choices.

The target variable, NObeyesdad, is a categorical variable with seven classes:

- Insufficient Weight  
- Normal Weight  
- Overweight Level I  
- Overweight Level II  
- Obesity Type I  
- Obesity Type II  
- Obesity Type III  

This makes the task a multi-class classification problem. The goal of supervised learning here is to learn a mapping from the input features to the obesity class. This is useful because:

- Machine learning can detect subtle patterns between lifestyle variables and obesity level that may be hard for humans to quantify.
- Once trained, the model can make predictions quickly and cheaply for many individuals, saving time and cost compared to clinical evaluation.
- The same model can potentially be used to screen populations and identify high-risk individuals for early intervention.

In summary, this is a supervised multi-class classification problem where the target variable is the obesity level (`NObeyesdad`) and the input variables are the demographic and lifestyle features in the dataset.



Data Source: UCI Machine Learning Repository — "Estimation of Obesity Levels Based on Eating Habits and Physical Condition" dataset, containing 2,111 individuals from Mexico, Peru, and Colombia.


In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo
# fetch dataset 
estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition = fetch_ucirepo(id=544) 
  
# data (as pandas dataframes) 
X = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.features 
y = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.targets 
  
# metadata 
#print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.metadata) 
  



# Sanity check

In [4]:
X

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Female,21.000000,1.620000,64.000000,yes,no,2.0,3.0,Sometimes,no,2.000000,no,0.000000,1.000000,no,Public_Transportation
1,Female,21.000000,1.520000,56.000000,yes,no,3.0,3.0,Sometimes,yes,3.000000,yes,3.000000,0.000000,Sometimes,Public_Transportation
2,Male,23.000000,1.800000,77.000000,yes,no,2.0,3.0,Sometimes,no,2.000000,no,2.000000,1.000000,Frequently,Public_Transportation
3,Male,27.000000,1.800000,87.000000,no,no,3.0,3.0,Sometimes,no,2.000000,no,2.000000,0.000000,Frequently,Walking
4,Male,22.000000,1.780000,89.800000,no,no,2.0,1.0,Sometimes,no,2.000000,no,0.000000,0.000000,Sometimes,Public_Transportation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2106,Female,20.976842,1.710730,131.408528,yes,yes,3.0,3.0,Sometimes,no,1.728139,no,1.676269,0.906247,Sometimes,Public_Transportation
2107,Female,21.982942,1.748584,133.742943,yes,yes,3.0,3.0,Sometimes,no,2.005130,no,1.341390,0.599270,Sometimes,Public_Transportation
2108,Female,22.524036,1.752206,133.689352,yes,yes,3.0,3.0,Sometimes,no,2.054193,no,1.414209,0.646288,Sometimes,Public_Transportation
2109,Female,24.361936,1.739450,133.346641,yes,yes,3.0,3.0,Sometimes,no,2.852339,no,1.139107,0.586035,Sometimes,Public_Transportation


In [5]:
y

,NObeyesdad
0,Normal_Weight
1,Normal_Weight
2,Normal_Weight
3,Overweight_Level_I
4,Overweight_Level_II
...,...
2106,Obesity_Type_III
2107,Obesity_Type_III
2108,Obesity_Type_III
2109,Obesity_Type_III


In [6]:
# Check missing values
print(X.isnull().sum().sum())
print(y.isnull().sum())

0
NObeyesdad    0
dtype: int64


In [7]:
# checking data types
X.dtypes

Gender                             object
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight     object
FAVC                               object
FCVC                              float64
NCP                               float64
CAEC                               object
SMOKE                              object
CH2O                              float64
SCC                                object
FAF                               float64
TUE                               float64
CALC                               object
MTRANS                             object
dtype: object

In [8]:
y.dtypes

NObeyesdad    object
dtype: object

In [9]:
y = y["NObeyesdad"]   
print("\nTarget preview:")
print(y.head())



Target preview:
0          Normal_Weight
1          Normal_Weight
2          Normal_Weight
3     Overweight_Level_I
4    Overweight_Level_II
Name: NObeyesdad, dtype: object


In [10]:
# Train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (1688, 16)
X_test: (423, 16)


In [11]:
# Identify numeric and categorical predictors
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)



Numeric columns: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Categorical columns: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


# Exploratory Data Analysis

The dataset contains 2,111 rows (individuals) and 16 feature columns, plus a separate target column `NObeyesdad`. The features include both numeric variables (e.g., Age, Height, Weight,'FCVC', 'NCP' CH2O, FAF, TUE) and categorical variables (e.g., Gender, family_history_with_overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS).

# Meaning of abbreviations
- Frequent consumption of high caloric food (FAVC)
- Frequency of consumption of vegetables (FCVC)
- Number of main meals (NCP)
- Consumption of food between meals (CAEC)
- Consumption of water daily (CH20)
- Consumption of alcohol (CALC)
- Calories consumption monitoring (SCC)
- Physical activity frequency (FAF)
- Time using technology devices (TUE)
- Transportation used (MTRANS)


# Missing Values

A check of missing values shows:

- No missing values in any of the 16 feature columns.
- No missing values in the target variable `NObeyesdad`.

Because there is no missing data, I do not need a special imputation strategy beyond what is already handled inside the pipelines (using default `SimpleImputer`, which in this case is effectively a safeguard and not actually used).

# Numeric vs Categorical Features

From the dtypes:

- Numeric features include:  
  `Age`, `Height`, `Weight`, `FCVC`, `NCP`, `CH2O`, `FAF`, `TUE`.

- Categorical features include:  
  `Gender`, `family_history_with_overweight`, `FAVC`, `CAEC`, `SMOKE`, `SCC`, `CALC`, `MTRANS`.

Numeric features will be standardized using `StandardScaler`, while categorical features will be one-hot encoded with `OneHotEncoder` inside a `ColumnTransformer`.

# Target Variable

The target variable `NObeyesdad` has seven classes (Insufficient_Weight, Normal_Weight, Overweight_Level_I, Overweight_Level_II, Obesity_Type_I, Obesity_Type_II, Obesity_Type_III). The class counts and proportions show that the data is not perfectly balanced, but each class is reasonably represented. Because the classes are not equal in size, and because all classes are important from a health perspective, treating this as an imbalanced multi-class classification problem is appropriate.

# Outliers 

A summary of numeric variables shows:

- Ages are in a realistic range for adults.
- Heights and weights fall into plausible human ranges.
- Variables like `CH2O` (water consumption), `FAF` (physical activity frequency), and `TUE` (technology use time) do not show obvious impossible values.

Therefore, I do not remove any rows; instead, I allow the models to learn directly from the full dataset.


# Evaluation Metric

Since this is a multi-class classification problem and the classes are not perfectly balanced, I used the Macro F1 Score as my evaluation metric.

Macro F1 gives an F1 score for each class and then averages them.

I used Macro F1 both during cross-validation and when evaluating the final models on the test set.



In [12]:
numeric_transformer = Pipeline(steps=[
("imputer", SimpleImputer(strategy="median")),
("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
("imputer", SimpleImputer(strategy="most_frequent")),
("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
transformers=[
("num", numeric_transformer, numeric_cols),
("cat", categorical_transformer, categorical_cols)
]
)

In [13]:
# Logistic Regression 
log_clf = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    max_iter=5000,
    multi_class="multinomial"
)

log_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__l1_ratio": [0.0, 0.5, 1.0]
}

pipe_log = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", log_clf)
])


In [14]:
# SVM  
svm_clf = SVC(kernel="rbf")

svm_params = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", 0.01, 0.001]
}

pipe_svm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", svm_clf)
])

In [15]:
# Random Forest 
rf_clf = RandomForestClassifier(random_state=42)

rf_params = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__max_features": ["sqrt", "log2"]
}

pipe_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_clf)
])


In [16]:
scoring = "f1_macro"

models = {
    "LogisticRegression": (pipe_log, log_params),
    "SVM": (pipe_svm, svm_params),
    "RandomForest": (pipe_rf, rf_params)
}

best_models = {}

for name, (pipeline, params) in models.items():
    print(f"\n===== TUNING {name} =====")
    
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring=scoring,
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    print("Best Params:", grid.best_params_)
    print("Best CV f1_macro:", grid.best_score_)
    
    best_models[name] = grid.best_estimator_



===== TUNING LogisticRegression =====
Best Params: {'model__C': 10, 'model__l1_ratio': 1.0}
Best CV f1_macro: 0.9577283283504435

===== TUNING SVM =====
Best Params: {'model__C': 10, 'model__gamma': 'scale'}
Best CV f1_macro: 0.9464668752200158

===== TUNING RandomForest =====
Best Params: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__n_estimators': 500}
Best CV f1_macro: 0.9426343254165296


# Model Fitting

For this project, I trained three different supervised machine learning models to predict obesity levels using the features in the dataset. I chose models that represent different types of learning approaches so I could compare how linear, nonlinear, and tree-based methods perform on this classification task.

- I created a preprocessing pipeline that handles numerical scaling and categorical one-hot encoding.
- Each model was placed inside its own pipeline so that preprocessing happens automatically during training.
- I used 5-fold cross-validation with `GridSearchCV` to tune the hyperparameters for each model.
- The tuning was based on Macro F1 score, so the selected hyperparameters reflect the model that performs best across all obesity classes.
- After finding the best parameters, each model was refit on the full training set and then evaluated on the held-out test set.

This approach ensures that all models are trained consistently, avoids data leakage, and allows for a fair comparison of their performance.


In [17]:
def evaluate(name, model):
    preds = model.predict(X_test)
    f1 = f1_score(y_test, preds, average="macro")  
    
    print(f"\n===== {name} TEST RESULTS =====")
    print("Macro F1:", round(f1, 4))
    print("\nClassification Report:\n", classification_report(y_test, preds))
    
    return {"Model": name, "Macro_F1": round(f1, 4)}

results = [evaluate(name, model) for name, model in best_models.items()]
results_df = pd.DataFrame(results)
print(results_df)


===== LogisticRegression TEST RESULTS =====
Macro F1: 0.9659

Classification Report:
                      precision    recall  f1-score   support

Insufficient_Weight       0.96      1.00      0.98        54
      Normal_Weight       0.95      0.91      0.93        58
     Obesity_Type_I       0.99      0.99      0.99        70
    Obesity_Type_II       0.97      1.00      0.98        60
   Obesity_Type_III       1.00      0.98      0.99        65
 Overweight_Level_I       0.92      0.93      0.92        58
Overweight_Level_II       0.98      0.95      0.96        58

           accuracy                           0.97       423
          macro avg       0.97      0.97      0.97       423
       weighted avg       0.97      0.97      0.97       423


===== SVM TEST RESULTS =====
Macro F1: 0.9467

Classification Report:
                      precision    recall  f1-score   support

Insufficient_Weight       0.94      0.94      0.94        54
      Normal_Weight       0.87      0.90    

In [18]:
best_idx = results_df["Macro_F1"].idxmax()
best_model = results_df.loc[best_idx, "Model"]
best_score = results_df.loc[best_idx, "Macro_F1"]

print("\nBest Model:", best_model)
print("Best Test Macro F1:", best_score)



Best Model: LogisticRegression
Best Test Macro F1: 0.9659


# Model Comparison

All three models were tuned using 5-fold cross-validation and evaluated on the same test set. The results show clear differences in performance:

- Logistic Regression performed the best, with a Macro F1 and accuracy around 0.97. It handled all seven obesity classes well, including the smaller ones.
- SVM also performed strongly with a Macro F1 of about 0.95, but slightly below the logistic model.
- Random Forest achieved a Macro F1 of about 0.93, still solid but not as strong as the other two.

Overall, Logistic Regression offered the best balance of performance and simplicity. Despite being a linear model, the elastic net regularization allowed it to generalize well without overfitting. The SVM captured nonlinear patterns but did not outperform the simpler model, and the Random Forest, while flexible, did not provide additional gains on this dataset.

# Final Model  
Logistic Regression is selected as the final model due to its highest Macro F1 score and strong, consistent performance across all obesity categories.


# Conclusion

This project set out to determine whether obesity levels can be predicted from lifestyle and behavioral indicators. The results confirm that these variables contain substantial predictive information. Among the models evaluated, Logistic Regression with elastic net regularization achieved the highest Macro F1 score, indicating strong and balanced performance across all seven obesity categories.

The finding that a linear model outperformed both SVM and Random Forest suggests that the relationships in the data, while meaningful, follow a relatively stable structure that does not require highly nonlinear decision boundaries. This aligns with existing public health research emphasizing consistent behavioral predictors of obesity.

In conclusion, the study demonstrates that machine learning can support early identification of obesity risk using simple, accessible features. With further refinement and broader data collection, such models could contribute to population-level screening, prevention efforts, and resource allocation in health settings.